In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, json
import torch
import time
t0 = time.time()
import torch.nn.functional as F
from torchvision import datasets
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = "/content/drive/MyDrive/intel-image-classifier"
sys.path.insert(0, f"{PROJECT_ROOT}/src")
os.chdir(PROJECT_ROOT)
print("Connected to project folder.")

from preprocessing import get_transforms, MEAN, STD
from cnn_model import CNN
from resnet_model import build_resnet18

local_test = "/content/test_local"
if not os.path.exists(local_test):
    if os.path.exists("/content/data/seg_test/seg_test"):
        local_test = "/content/data/seg_test/seg_test"
    elif os.path.exists("data/seg_test/seg_test"):
        print(f"[{time.time()-t0:.1f}s] Copying test images from Drive to fast local disk (once)...")
        shutil.copytree("data/seg_test/seg_test", local_test)
print(f"[{time.time()-t0:.1f}s] Using test folder: {local_test}")

def denormalize(tensor):
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)
    return torch.clamp(tensor.cpu() * std + mean, 0, 1).permute(1, 2, 0).numpy()

def evaluate_model(model_type, checkpoint_path, test_dir, device):
    print(f"\n[{time.time()-t0:.1f}s] Loading {model_type.upper()}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    class_names = checkpoint['class_names']
    img_size = checkpoint['img_size']

    model = CNN(num_classes=len(class_names)) if model_type == "cnn" else build_resnet18(num_classes=len(class_names), freeze_backbone=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device).eval()

    _, test_tf = get_transforms(img_size=img_size)
    test_dataset = datasets.ImageFolder(root=test_dir, transform=test_tf)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

    all_preds, all_labels, misclassified = [], [], []
    with torch.inference_mode():
        for batch_idx, (images, labels) in enumerate(test_loader):
            outputs = model(images.to(device))
            probs = F.softmax(outputs, dim=1)
            confidences, preds = torch.max(probs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            if batch_idx % 8 == 0:
                print(f"[{time.time()-t0:.1f}s] Batch {batch_idx+1}/{len(test_loader)} done")

            if len(misclassified) < 10:
                for i in range(len(labels)):
                    if preds[i].item() != labels[i].item() and len(misclassified) < 10:
                        misclassified.append({
                            'image': images[i].cpu(),
                            'actual': class_names[labels[i].item()],
                            'predicted': class_names[preds[i].item()],
                            'confidence': confidences[i].item()
                        })

    acc = accuracy_score(all_labels, all_preds)
    print(f"[{time.time()-t0:.1f}s] {model_type.upper()} Test Accuracy: {acc:.4f}\n")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

    os.makedirs("outputs/metrics", exist_ok=True)
    with open(f"outputs/metrics/{model_type}_test_report.json", "w") as f:
        json.dump(classification_report(all_labels, all_preds, target_names=class_names, output_dict=True), f, indent=4)

    os.makedirs("outputs/confusion_matrices", exist_ok=True)
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {model_type.upper()}')
    plt.xlabel('Predicted Class'); plt.ylabel('Actual Class')
    plt.tight_layout()
    plt.savefig(f"outputs/confusion_matrices/{model_type}_cm.png")
    plt.show()

    os.makedirs("outputs/plots", exist_ok=True)
    if misclassified:
        plt.figure(figsize=(15, 6))
        for idx, item in enumerate(misclassified):
            plt.subplot(2, 5, idx + 1)
            plt.imshow(denormalize(item['image']))
            plt.title(f"Act: {item['actual']}\nPred: {item['predicted']} ({item['confidence']:.1%})", fontsize=9)
            plt.axis('off')
        plt.suptitle(f"Top 10 Misclassified Test Images ({model_type.upper()})", fontsize=12)
        plt.tight_layout()
        plt.savefig(f"outputs/plots/{model_type}_misclassified.png")
        plt.show()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if os.path.exists("models/cnn_best.pth"):
    evaluate_model("cnn", "models/cnn_best.pth", local_test, device)
if os.path.exists("models/resnet18_best.pth"):
    evaluate_model("resnet18", "models/resnet18_best.pth", local_test, device)

Output hidden; open in https://colab.research.google.com to view.